# Mohan Sir — Head and Shoulder Landmark Pilot

**EXPERIMENTAL ONLY.** This notebook prepares a bounded 2D pose-landmark preview; it does not produce a 3D avatar, facial-expression analysis, 3D head pose, a validated pose track, retargeting, training, or a render. It uses independent OpenPose heatmap peaks and intentionally does not perform PAF-based person matching. Treat every coordinate as unvalidated until visually reviewed.

The source is chest-up, so this cannot recover full-body motion. Hands/arms are mostly outside the frame, and the inset near 00:24–00:25 is excluded by processing only 00:00–00:20. The only output is a local CSV and an optional in-notebook overlay. No Bunny writes or uploads occur.

## Scope, permission, and model rights

Process only the owner-approved Marathi October 26 Speaker View video (`3246a191-7f93-4002-ab2d-4d7916410577`, Bunny library `606386`), and only its first 20 seconds at 1 frame per second. The HLS stream is read from Bunny CDN into this Colab runtime; do not use this notebook with another source or broaden the time range. No Bunny API key is needed.

Model candidate: OpenVINO Model Zoo `human-pose-estimation-0001`, pinned to repository revision `af810de42bcf460a18610481f639dcdc86249e69`. Its model descriptor declares the Apache-2.0 license and provides checksums for the FP16 artifacts; this notebook verifies both size and SHA-384 before loading. The official model documentation describes 18 2D body keypoints and reports 42.8 AP on a COCO validation subset. Training-data provenance is not fully described, so this is for exploratory review only, not a production/commercial rights certification.

References: [pinned model descriptor](https://github.com/openvinotoolkit/open_model_zoo/blob/af810de42bcf460a18610481f639dcdc86249e69/models/intel/human-pose-estimation-0001/model.yml), [Apache-2.0 license at the same revision](https://github.com/openvinotoolkit/open_model_zoo/blob/af810de42bcf460a18610481f639dcdc86249e69/LICENSE), [model capabilities and evaluation](https://docs.openvino.ai/2023.3/omz_models_model_human_pose_estimation_0001.html), [OpenVINO runtime package](https://pypi.org/project/openvino/2025.4.0/).

## 1. Explicit authorization gate

The next cell stops by default. Confirm the already-approved source and this exact 20-second scope before changing the flags. The following cells install a pinned runtime, download model files, and process the source.

In [ ]:
PROCESSING_AUTHORIZED = False
SOURCE_AND_TIME_RANGE_APPROVED = False
MODEL_LICENSE_REVIEWED = False

if not all((PROCESSING_AUTHORIZED, SOURCE_AND_TIME_RANGE_APPROVED, MODEL_LICENSE_REVIEWED)):
    raise RuntimeError('Stopped: confirm all three authorization gates before installing or processing anything.')

print('Authorization gates confirmed for this Colab session only.')

## 2. Install the pinned CPU runtime

This installs only OpenVINO `2025.4.0` from PyPI after the authorization gate. Inference is CPU-only; no NVIDIA GPU is required. The model weights are fetched separately and hash-checked in the next section.

In [ ]:
if not all((PROCESSING_AUTHORIZED, SOURCE_AND_TIME_RANGE_APPROVED, MODEL_LICENSE_REVIEWED)):
    raise RuntimeError('Stopped: confirm authorization before installing the runtime.')

import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'openvino==2025.4.0'], check=True)
import openvino as ov

if not ov.__version__.startswith('2025.4.0'):
    raise RuntimeError(f'Unexpected OpenVINO version: {ov.__version__}')
print('OpenVINO:', ov.__version__, '| device: CPU')

## 3. Download and verify the pinned FP16 model artifacts

Only the two FP16 files listed in the pinned model descriptor are downloaded. A size or SHA-384 mismatch stops execution before inference.

In [ ]:
if not all((PROCESSING_AUTHORIZED, SOURCE_AND_TIME_RANGE_APPROVED, MODEL_LICENSE_REVIEWED)):
    raise RuntimeError('Stopped: confirm authorization before downloading model files.')

import hashlib
from pathlib import Path
from urllib.request import Request, urlopen

MODEL_DIR = Path('/content/mohan_pose_model')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_FILES = {
    'human-pose-estimation-0001.xml': {
        'url': 'https://storage.openvinotoolkit.org/repositories/open_model_zoo/2023.0/models_bin/1/human-pose-estimation-0001/FP16/human-pose-estimation-0001.xml',
        'size': 218215,
        'sha384': 'cffe8df7d053b9cbf858a21faa32e30cb8a645416e9ee4ce3fbcc3106094505477d66a06b7e46d6ddb9c2de4b0cee319',
    },
    'human-pose-estimation-0001.bin': {
        'url': 'https://storage.openvinotoolkit.org/repositories/open_model_zoo/2023.0/models_bin/1/human-pose-estimation-0001/FP16/human-pose-estimation-0001.bin',
        'size': 8197354,
        'sha384': 'dabb7be42c5be008de354c6670aa8291c2d18e59f18a1c138df9b5200929a03c1e323930a1d21ecff69d3f06007de67c',
    },
}

def download_verified(name, details):
    destination = MODEL_DIR / name
    request = Request(details['url'], headers={'User-Agent': 'SwarYoga-HeadShoulder-Pilot/1.0'})
    digest = hashlib.sha384()
    byte_count = 0
    with urlopen(request, timeout=60) as response, destination.open('wb') as output:
        while True:
            chunk = response.read(1024 * 1024)
            if not chunk:
                break
            byte_count += len(chunk)
            if byte_count > details['size']:
                raise RuntimeError(f'{name}: download exceeded the pinned size')
            digest.update(chunk)
            output.write(chunk)
    if byte_count != details['size'] or digest.hexdigest() != details['sha384']:
        destination.unlink(missing_ok=True)
        raise RuntimeError(f'{name}: pinned size/SHA-384 verification failed')
    print(f'Verified {name}: {byte_count} bytes; SHA-384 {digest.hexdigest()}')
    return destination

MODEL_PATHS = {name: download_verified(name, details) for name, details in MODEL_FILES.items()}

## 4. Decode only the approved 20 seconds

FFmpeg must already be present in the Colab runtime. It is not installed here. The command caps decoding at 20 seconds, samples at 1 FPS, and keeps the 16:9 image ratio while resizing/padding to the model's `456×256` BGR input. Decoded frames remain in memory; no source clip is saved.

In [ ]:
if not all((PROCESSING_AUTHORIZED, SOURCE_AND_TIME_RANGE_APPROVED, MODEL_LICENSE_REVIEWED)):
    raise RuntimeError('Stopped: confirm authorization before reading the Bunny stream.')

import shutil
import subprocess
import numpy as np

HLS_URL = 'https://vz-d0769ece-079.b-cdn.net/3246a191-7f93-4002-ab2d-4d7916410577/1080p/video.m3u8'
REFERER = 'https://player.mediadelivery.net/'
SOURCE_WIDTH = 1920
SOURCE_HEIGHT = 1080
MODEL_WIDTH = 456
MODEL_HEIGHT = 256
RESIZED_WIDTH = 455
FRAME_COUNT = 20

ffmpeg = shutil.which('ffmpeg')
if ffmpeg is None:
    raise RuntimeError('FFmpeg is missing; stop rather than installing unreviewed media software.')

command = [
    ffmpeg, '-nostdin', '-hide_banner', '-loglevel', 'error',
    '-headers', f'Referer: {REFERER}\r\n', '-i', HLS_URL,
    '-t', '20', '-vf', 'fps=1,scale=455:256:flags=area,pad=456:256:0:0:color=black,format=bgr24',
    '-frames:v', str(FRAME_COUNT), '-f', 'rawvideo', 'pipe:1',
]
decoded = subprocess.run(command, capture_output=True, timeout=180, check=True)
bytes_per_frame = MODEL_WIDTH * MODEL_HEIGHT * 3
expected_bytes = FRAME_COUNT * bytes_per_frame
if len(decoded.stdout) != expected_bytes:
    raise RuntimeError(f'Expected exactly {FRAME_COUNT} frames; got {len(decoded.stdout) // bytes_per_frame}. Stop for review.')
frames_bgr = np.frombuffer(decoded.stdout, dtype=np.uint8).reshape(FRAME_COUNT, MODEL_HEIGHT, MODEL_WIDTH, 3).copy()
del decoded
print(f'Decoded {len(frames_bgr)} frames, t=0..19 seconds; no video file was saved.')

## 5. Produce raw 2D joint candidates

The model receives BGR `0–255` pixels in NCHW layout, as documented by Open Model Zoo. The CSV stores the strongest heatmap cell for each joint. These are **raw heatmap peaks, not calibrated confidence values**. PAF association is deliberately omitted, so this is not reliable person tracking; it is a bounded visual-review aid only.

In [ ]:
if not all((PROCESSING_AUTHORIZED, SOURCE_AND_TIME_RANGE_APPROVED, MODEL_LICENSE_REVIEWED)):
    raise RuntimeError('Stopped: confirm authorization before model inference.')

import csv

JOINT_NAMES = [
    'nose', 'neck', 'right_shoulder', 'right_elbow', 'right_wrist',
    'left_shoulder', 'left_elbow', 'left_wrist', 'right_hip', 'right_knee',
    'right_ankle', 'left_hip', 'left_knee', 'left_ankle', 'right_eye',
    'left_eye', 'right_ear', 'left_ear',
]
core = ov.Core()
model = core.read_model(str(MODEL_PATHS['human-pose-estimation-0001.xml']))
input_shape = tuple(int(dim) for dim in model.input(0).shape)
if input_shape != (1, 3, MODEL_HEIGHT, MODEL_WIDTH):
    raise RuntimeError(f'Unexpected model input shape: {input_shape}')
compiled = core.compile_model(model, 'CPU')
heatmap_port = next((port for port in compiled.outputs if 'Mconv7_stage2_L2' in port.any_name), None)
if heatmap_port is None:
    raise RuntimeError('Expected OpenPose heatmap output was not found')

rows = []
heatmap_frames = []
for frame_index, frame in enumerate(frames_bgr):
    tensor = frame.transpose(2, 0, 1)[None].astype(np.float32)
    result = compiled({compiled.input(0): tensor})
    heatmaps = np.asarray(result[heatmap_port])[0]
    if heatmaps.shape != (19, 32, 57):
        raise RuntimeError(f'Unexpected heatmap shape: {heatmaps.shape}')
    heatmap_frames.append(heatmaps)
    for joint_index, joint_name in enumerate(JOINT_NAMES):
        joint_map = heatmaps[joint_index]
        flat_index = int(np.argmax(joint_map))
        grid_y, grid_x = np.unravel_index(flat_index, joint_map.shape)
        peak = float(joint_map[grid_y, grid_x])
        resized_x = grid_x * 8.0
        resized_y = grid_y * 8.0
        rows.append({
            'frame_index': frame_index,
            'time_seconds': frame_index,
            'joint': joint_name,
            'grid_x': int(grid_x),
            'grid_y': int(grid_y),
            'source_x_px': round(resized_x * SOURCE_WIDTH / RESIZED_WIDTH, 2),
            'source_y_px': round(resized_y * SOURCE_HEIGHT / MODEL_HEIGHT, 2),
            'raw_heatmap_peak_not_confidence': peak,
        })

CSV_PATH = Path('/content/mohan_sir_head_shoulder_pose_preview.csv')
with CSV_PATH.open('w', newline='', encoding='utf-8') as output:
    writer = csv.DictWriter(output, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)
print(f'Wrote {len(rows)} raw 2D joint candidates to {CSV_PATH}; no render or Bunny write occurred.')

## 6. Human visual review

The overlay is a diagnostic of independent heatmap peaks, not a completed skeleton. Check whether nose, neck, and shoulders land on Mohan Sir consistently. Do not treat lower-body points or cropped/occluded wrists as valid motion. If the dots jump to the background or another person, reject this pilot output.

In [ ]:
import matplotlib.pyplot as plt

preview_frames = [0, 5, 10, 15]
figure, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, frame_index in zip(axes.flat, preview_frames):
    ax.imshow(frames_bgr[frame_index, :, :RESIZED_WIDTH, ::-1])
    for joint_index, joint_name in enumerate(JOINT_NAMES):
        joint_map = heatmap_frames[frame_index][joint_index]
        grid_y, grid_x = np.unravel_index(int(np.argmax(joint_map)), joint_map.shape)
        ax.scatter(grid_x * 8, grid_y * 8, s=12)
        if joint_index in (0, 1, 2, 5):
            ax.text(grid_x * 8 + 2, grid_y * 8, joint_name, fontsize=6, color='yellow')
    ax.set_title(f't={frame_index}s — raw 2D peaks')
    ax.axis('off')
plt.tight_layout()
plt.show()
print('Review the dots against the approved source. This overlay is not a validated pose or avatar output.')

## 7. Optional manual CSV download and cleanup

The CSV remains only in the current Colab runtime unless you choose to download it. Model files and frames are temporary. No output is sent to Bunny or another service.

In [ ]:
try:
    from google.colab import files
    files.download(str(CSV_PATH))
except ImportError:
    print('Not running in Colab; CSV is at', CSV_PATH)

for path in MODEL_DIR.iterdir():
    if path.is_file():
        path.unlink()
MODEL_DIR.rmdir()
del frames_bgr, heatmap_frames, compiled, model, core
print('Temporary model files and in-memory source frames cleared; CSV retained for your review.')